# Experiment 22: Heavy Ensemble

This experiment builds a stronger Titanic ensemble using repeated stratified cross-validation, multiple XGBoost and CatBoost configurations, Extra Trees, Random Forest, out-of-fold predictions, weighted blending, and stacking.

The goal is to test whether a diverse ensemble can outperform the individual models while keeping validation leakage controlled.

No external survival labels are used.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 2

BASE_DIR = Path('C:/Users/aakif/Documents/DataCompetition2')
DATA_DIR = BASE_DIR / 'data'

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
print('Target:     ', 'Survived')

Train shape: (891, 12)
Test shape:  (418, 11)
Target:      Survived


## Feature engineering

The feature builder keeps the strongest passenger, family, ticket, fare, title, cabin, and interaction features used in earlier experiments.

Group counts are calculated from the supplied reference dataframe. During cross-validation, validation rows are never used to calculate their group counts.

In [2]:
def clean_title(title):
    title = str(title).strip()
    title = {'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'}.get(title, title)
    common = {'Mr', 'Miss', 'Mrs', 'Master'}
    return title if title in common else 'Rare'


def engineer_features(df, reference=None):
    out = df.copy()
    ref = out if reference is None else reference.copy()

    out['FamilySize'] = out['SibSp'] + out['Parch'] + 1
    out['IsAlone'] = (out['FamilySize'] == 1).astype(int)

    out['Title'] = out['Name'].str.extract(r',\\s*([^.]*)\\.', expand=False).fillna('Rare').map(clean_title)
    out['Surname'] = out['Name'].str.split(',').str[0].str.strip()

    out['CabinDeck'] = out['Cabin'].fillna('Unknown').astype(str).str[0]
    out['HasCabin'] = out['Cabin'].notna().astype(int)
    out['DeckKnown'] = out['HasCabin']
    out['CabinCount'] = out['Cabin'].fillna('').astype(str).str.count(' ') + out['HasCabin']

    out['TicketPrefix'] = (
        out['Ticket'].astype(str)
        .str.replace(r'\\d', '', regex=True)
        .str.replace(r'[./]', '', regex=True)
        .str.replace(' ', '', regex=True)
        .replace('', 'NONE')
    )

    ref['Surname'] = ref['Name'].str.split(',').str[0].str.strip()
    out['TicketGroupSize'] = out['Ticket'].map(ref['Ticket'].value_counts()).fillna(1)
    out['SurnameGroupSize'] = out['Surname'].map(ref['Surname'].value_counts()).fillna(1)

    out['FarePerPerson'] = out['Fare'] / out['TicketGroupSize'].replace(0, 1)
    out['FarePerAge'] = out['Fare'] / out['Age'].clip(lower=1)
    out['ClassFare'] = out['Pclass'] * out['Fare']
    out['FamilyFare'] = out['Fare'] / out['FamilySize'].replace(0, 1)

    out['AgeMissing'] = out['Age'].isna().astype(int)
    out['FareMissing'] = out['Fare'].isna().astype(int)
    out['EmbarkedMissing'] = out['Embarked'].isna().astype(int)
    out['FarePerPersonMissing'] = out['FarePerPerson'].isna().astype(int)

    out['Child'] = ((out['Age'] < 16) & out['Age'].notna()).astype(int)
    out['Mother'] = ((out['Sex'] == 'female') & (out['Parch'] > 0) & (out['Age'] > 18)).astype(int)
    out['LargeFamily'] = (out['FamilySize'] >= 5).astype(int)
    out['SmallFamily'] = out['FamilySize'].between(2, 4).astype(int)
    out['FemaleChild'] = ((out['Sex'] == 'female') & (out['Age'] < 16)).astype(int)

    out['Sex_Pclass'] = out['Sex'].astype(str) + '_' + out['Pclass'].astype(str)
    out['FamilySex'] = out['Sex'].astype(str) + '_' + out['FamilySize'].astype(str)
    out['PclassTitle'] = out['Pclass'].astype(str) + '_' + out['Title'].astype(str)
    out['FamilyTicket'] = out['FamilySize'].astype(str) + '_' + out['TicketPrefix'].astype(str)
    out['SexTitle'] = out['Sex'].astype(str) + '_' + out['Title'].astype(str)

    out['SiblingChildRatio'] = out['SibSp'] / (out['Parch'] + 1)
    out['NameLength'] = out['Name'].astype(str).str.len()
    out['NameWords'] = out['Name'].astype(str).str.split().str.len()
    out['TicketLength'] = out['Ticket'].astype(str).str.len()

    out['FamilySizeBand'] = pd.cut(
        out['FamilySize'],
        bins=[0, 1, 4, 7, np.inf],
        labels=['Alone', 'Small', 'Medium', 'Large']
    ).astype(str)
    out['AgeBand'] = pd.cut(
        out['Age'],
        bins=[-np.inf, 5, 12, 18, 30, 45, 60, np.inf],
        labels=['Baby', 'Child', 'Teen', 'YoungAdult', 'Adult', 'Mature', 'Senior']
    ).astype(str)
    out['FareBand'] = pd.qcut(out['Fare'], q=5, duplicates='drop').astype(str)

    return out


TARGET = 'Survived'
y = train[TARGET].astype(int).to_numpy()
X_raw = train.drop(columns=[TARGET])
test_raw = test.copy()

## Preprocessing for tree ensembles

The one-hot pipeline is used for XGBoost, Extra Trees, and Random Forest. The CatBoost models use the categorical variables directly.

In [3]:
def prepare_tree_data(train_part, valid_part, test_part=None):
    train_eng = engineer_features(train_part, train_part)
    valid_eng = engineer_features(valid_part, train_part)
    test_eng = engineer_features(test_part, train_part) if test_part is not None else None

    drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Surname']
    train_eng = train_eng.drop(columns=[c for c in drop_cols if c in train_eng.columns])
    valid_eng = valid_eng.drop(columns=[c for c in drop_cols if c in valid_eng.columns])
    if test_eng is not None:
        test_eng = test_eng.drop(columns=[c for c in drop_cols if c in test_eng.columns])

    categorical = train_eng.select_dtypes(include=['object', 'category']).columns.tolist()
    numeric = [c for c in train_eng.columns if c not in categorical]

    preprocessor = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), numeric),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical)
    ], remainder='drop')

    train_matrix = preprocessor.fit_transform(train_eng)
    valid_matrix = preprocessor.transform(valid_eng)
    test_matrix = preprocessor.transform(test_eng) if test_eng is not None else None

    return train_matrix, valid_matrix, test_matrix, train_eng, valid_eng, test_eng


def prepare_catboost_data(train_part, valid_part, test_part=None):
    train_eng = engineer_features(train_part, train_part)
    valid_eng = engineer_features(valid_part, train_part)
    test_eng = engineer_features(test_part, train_part) if test_part is not None else None

    drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Surname']
    train_eng = train_eng.drop(columns=[c for c in drop_cols if c in train_eng.columns])
    valid_eng = valid_eng.drop(columns=[c for c in drop_cols if c in valid_eng.columns])
    if test_eng is not None:
        test_eng = test_eng.drop(columns=[c for c in drop_cols if c in test_eng.columns])

    categorical = train_eng.select_dtypes(include=['object', 'category']).columns.tolist()
    for frame in [train_eng, valid_eng] + ([test_eng] if test_eng is not None else []):
        for col in categorical:
            frame[col] = frame[col].astype(str).fillna('Missing')

    train_eng[categorical] = train_eng[categorical].fillna('Missing')
    valid_eng[categorical] = valid_eng[categorical].fillna('Missing')
    if test_eng is not None:
        test_eng[categorical] = test_eng[categorical].fillna('Missing')

    return train_eng, valid_eng, test_eng, categorical

## Base model definitions

We deliberately use different model families and configurations. The point is not simply maximum individual accuracy. The point is to generate strong but different predictions that can be combined.

In [4]:
def make_xgb_models():
    common = dict(
        objective='binary:logistic',
        eval_metric='logloss',
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
    return {
        'XGB_A': XGBClassifier(
            n_estimators=1400, max_depth=3, learning_rate=0.018,
            subsample=0.85, colsample_bytree=0.85, min_child_weight=2,
            gamma=0.05, reg_alpha=0.05, reg_lambda=2.0, **common
        ),
        'XGB_B': XGBClassifier(
            n_estimators=1100, max_depth=4, learning_rate=0.022,
            subsample=0.80, colsample_bytree=0.90, min_child_weight=3,
            gamma=0.10, reg_alpha=0.10, reg_lambda=2.5, **common
        ),
        'XGB_C': XGBClassifier(
            n_estimators=1600, max_depth=2, learning_rate=0.018,
            subsample=0.90, colsample_bytree=0.80, min_child_weight=1,
            gamma=0.0, reg_alpha=0.0, reg_lambda=3.0, **common
        )
    }


def make_cat_models():
    return {
        'CAT_A': CatBoostClassifier(
            iterations=1200, depth=5, learning_rate=0.025,
            loss_function='Logloss', l2_leaf_reg=6,
            random_seed=RANDOM_STATE, verbose=False
        ),
        'CAT_B': CatBoostClassifier(
            iterations=1000, depth=6, learning_rate=0.025,
            loss_function='Logloss', l2_leaf_reg=8,
            random_seed=RANDOM_STATE + 7, verbose=False
        ),
        'CAT_C': CatBoostClassifier(
            iterations=1500, depth=4, learning_rate=0.018,
            loss_function='Logloss', l2_leaf_reg=4,
            random_seed=RANDOM_STATE + 13, verbose=False
        )
    }


def make_other_models():
    return {
        'EXTRA': ExtraTreesClassifier(
            n_estimators=1200, max_features=0.75, min_samples_leaf=1,
            class_weight=None, random_state=RANDOM_STATE, n_jobs=-1
        ),
        'RF': RandomForestClassifier(
            n_estimators=1200, max_features=0.65, min_samples_leaf=1,
            max_depth=None, random_state=RANDOM_STATE, n_jobs=-1
        )
    }

## Repeated out-of-fold training

This is the expensive part. Each base model is evaluated through 5 folds repeated twice. Every validation prediction is generated by a model that did not train on that validation row.

The OOF predictions are then used for the blend and stacking stages.

In [5]:
cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

model_names = list(make_xgb_models()) + list(make_cat_models()) + list(make_other_models())
oof_sum = {name: np.zeros(len(train), dtype=float) for name in model_names}
oof_count = np.zeros(len(train), dtype=int)
fold_scores = {name: [] for name in model_names}

for split_number, (tr_idx, va_idx) in enumerate(cv.split(X_raw, y), start=1):
    print(f'\========== SPLIT {split_number}/{N_SPLITS * N_REPEATS} ==========')

    tr_df = X_raw.iloc[tr_idx].copy()
    va_df = X_raw.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    Xtr, Xva, _, _, _, _ = prepare_tree_data(tr_df, va_df)
    cat_tr, cat_va, _, cat_cols = prepare_catboost_data(tr_df, va_df)

    xgb_models = make_xgb_models()
    cat_models = make_cat_models()
    other_models = make_other_models()

    for name, model in xgb_models.items():
        model.fit(Xtr, y_tr)
        pred = model.predict_proba(Xva)[:, 1]
        oof_sum[name][va_idx] += pred
        score = accuracy_score(y_va, (pred >= 0.5).astype(int))
        fold_scores[name].append(score)
        print(f'{name:8s}: {score:.4f}')

    for name, model in cat_models.items():
        model.fit(cat_tr, y_tr, cat_features=cat_cols)
        pred = model.predict_proba(cat_va)[:, 1]
        oof_sum[name][va_idx] += pred
        score = accuracy_score(y_va, (pred >= 0.5).astype(int))
        fold_scores[name].append(score)
        print(f'{name:8s}: {score:.4f}')

    for name, model in other_models.items():
        model.fit(Xtr, y_tr)
        pred = model.predict_proba(Xva)[:, 1]
        oof_sum[name][va_idx] += pred
        score = accuracy_score(y_va, (pred >= 0.5).astype(int))
        fold_scores[name].append(score)
        print(f'{name:8s}: {score:.4f}')

    oof_count[va_idx] += 1

oof = pd.DataFrame({name: oof_sum[name] / oof_count for name in model_names})
oof['y'] = y

print('\OOF prediction matrix:', oof.shape)

\========== SPLIT 1/10 ==========
XGB_A   : 0.8603
XGB_B   : 0.8492
XGB_C   : 0.8547
CAT_A   : 0.8324
CAT_B   : 0.8492
CAT_C   : 0.8492
EXTRA   : 0.8268
RF      : 0.8492
\========== SPLIT 2/10 ==========
XGB_A   : 0.8371
XGB_B   : 0.8427
XGB_C   : 0.8371
CAT_A   : 0.8539
CAT_B   : 0.8258
CAT_C   : 0.8146
EXTRA   : 0.8258
RF      : 0.8596
\========== SPLIT 3/10 ==========
XGB_A   : 0.8146
XGB_B   : 0.8090
XGB_C   : 0.8202
CAT_A   : 0.8146
CAT_B   : 0.8146
CAT_C   : 0.7921
EXTRA   : 0.7978
RF      : 0.8034
\========== SPLIT 4/10 ==========
XGB_A   : 0.8315
XGB_B   : 0.8427
XGB_C   : 0.8202
CAT_A   : 0.8258
CAT_B   : 0.8427
CAT_C   : 0.8258
EXTRA   : 0.8258
RF      : 0.8258
\========== SPLIT 5/10 ==========
XGB_A   : 0.8371
XGB_B   : 0.8427
XGB_C   : 0.8315
CAT_A   : 0.8202
CAT_B   : 0.8483
CAT_C   : 0.8371
EXTRA   : 0.8483
RF      : 0.8539
\========== SPLIT 6/10 ==========
XGB_A   : 0.8212
XGB_B   : 0.8212
XGB_C   : 0.8324
CAT_A   : 0.8324
CAT_B   : 0.8156
CAT_C   : 0.7989
EXTRA   : 0.81

## Individual model results

In [6]:
individual_results = []

for name in model_names:
    pred = (oof[name].to_numpy() >= 0.5).astype(int)
    individual_results.append({
        'Model': name,
        'OOF Accuracy': accuracy_score(y, pred),
        'Mean Fold Accuracy': np.mean(fold_scores[name]),
        'Fold Std': np.std(fold_scores[name]),
        'Min Fold': np.min(fold_scores[name]),
        'Max Fold': np.max(fold_scores[name])
    })

individual_results = pd.DataFrame(individual_results).sort_values('OOF Accuracy', ascending=False)
display(individual_results)

,Model,OOF Accuracy,Mean Fold Accuracy,Fold Std,Min Fold,Max Fold
3,CAT_A,0.837262,0.829402,0.015762,0.808989,0.859551
4,CAT_B,0.837262,0.830525,0.017549,0.792135,0.849162
7,RF,0.832772,0.829402,0.023736,0.780899,0.859551
0,XGB_A,0.829405,0.827707,0.019884,0.780899,0.860335
2,XGB_C,0.829405,0.830513,0.015783,0.797753,0.854749
1,XGB_B,0.828283,0.831084,0.024046,0.769663,0.853933
5,CAT_C,0.824916,0.826602,0.024364,0.792135,0.859551
6,EXTRA,0.820426,0.815934,0.015872,0.792135,0.848315


## Search for strong weighted blends

The blend search uses the OOF probabilities only. No hidden test labels are used.

In [7]:
candidate_names = individual_results['Model'].tolist()
top_names = candidate_names[:6]

blend_results = []

# Equal-weight combinations of the strongest models.
from itertools import combinations

for size in range(2, min(6, len(top_names)) + 1):
    for combo in combinations(top_names, size):
        pred = oof[list(combo)].mean(axis=1).to_numpy()
        score = accuracy_score(y, (pred >= 0.5).astype(int))
        blend_results.append({
            'Models': ' + '.join(combo),
            'Accuracy': score,
            'Weights': 'equal'
        })

# A small systematic weight grid for the strongest three models.
if len(top_names) >= 3:
    a, b, c = top_names[:3]
    for wa in np.arange(0.1, 0.71, 0.1):
        for wb in np.arange(0.1, 0.71, 0.1):
            wc = 1.0 - wa - wb
            if wc < 0.1 or wc > 0.7:
                continue
            pred = wa * oof[a] + wb * oof[b] + wc * oof[c]
            score = accuracy_score(y, (pred >= 0.5).astype(int))
            blend_results.append({
                'Models': f'{a} + {b} + {c}',
                'Accuracy': score,
                'Weights': f'{wa:.1f}/{wb:.1f}/{wc:.1f}'
            })

blend_results = pd.DataFrame(blend_results).sort_values('Accuracy', ascending=False)
display(blend_results.head(20))

,Models,Accuracy,Weights
0,CAT_A + CAT_B,0.840629,equal
16,CAT_A + CAT_B + XGB_A,0.839506,equal
19,CAT_A + RF + XGB_A,0.839506,equal
67,CAT_A + CAT_B + RF,0.839506,0.2/0.6/0.2
61,CAT_A + CAT_B + RF,0.838384,0.1/0.6/0.3
37,CAT_A + CAT_B + RF + XGB_B,0.838384,equal
60,CAT_A + CAT_B + RF,0.838384,0.1/0.5/0.4
70,CAT_A + CAT_B + RF,0.838384,0.3/0.3/0.4
42,CAT_A + RF + XGB_A + XGB_B,0.837262,equal
21,CAT_A + RF + XGB_B,0.837262,equal


## Stacking

A logistic meta-model learns how to combine the base-model probabilities. Because it is trained only on OOF predictions, it does not directly see the training labels through predictions from models trained on those same rows.

In [8]:
stack_X = oof[model_names].copy()

meta_model = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(C=0.5, max_iter=5000, random_state=RANDOM_STATE))
])

meta_model.fit(stack_X, y)
stack_pred = meta_model.predict_proba(stack_X)[:, 1]
stack_score = accuracy_score(y, (stack_pred >= 0.5).astype(int))

print(f'Stacking OOF accuracy: {stack_score:.4f}')

Stacking OOF accuracy: 0.8350


## Final validation summary

In [9]:
best_single = individual_results.iloc[0]
best_blend = blend_results.iloc[0]

summary = pd.DataFrame([
    {
        'Approach': f"Best single: {best_single['Model']}",
        'Accuracy': best_single['OOF Accuracy']
    },
    {
        'Approach': f"Best blend: {best_blend['Models']} ({best_blend['Weights']})",
        'Accuracy': best_blend['Accuracy']
    },
    {
        'Approach': 'Stacking meta-model',
        'Accuracy': stack_score
    }
]).sort_values('Accuracy', ascending=False)

display(summary)
print('\Best approach:', summary.iloc[0]['Approach'])
print('Best OOF accuracy:', f"{summary.iloc[0]['Accuracy']:.4f}")

,Approach,Accuracy
1,Best blend: CAT_A + CAT_B (equal),0.840629
0,Best single: CAT_A,0.837262
2,Stacking meta-model,0.835017


\Best approach: Best blend: CAT_A + CAT_B (equal)
Best OOF accuracy: 0.8406


## Train the winning ensemble on all training data

After validation, the strongest ensemble is retrained on all 891 training passengers. This section prepares predictions for the Kaggle test set but does not create a submission automatically.

In [10]:
best_models = top_names[:3]
best_blend_row = blend_results.iloc[0]

print('Candidate models for final training:', best_models)
print('Best blend found:', best_blend_row.to_dict())

X_full, X_test, _, full_eng, test_eng, _ = prepare_tree_data(X_raw, test_raw, test_raw)
cat_full, cat_test, _, cat_cols = prepare_catboost_data(X_raw, test_raw, test_raw)

final_predictions = {}

for name in best_models:
    if name.startswith('XGB'):
        model = make_xgb_models()[name]
        model.fit(X_full, y)
        final_predictions[name] = model.predict_proba(X_test)[:, 1]
    elif name.startswith('CAT'):
        model = make_cat_models()[name]
        model.fit(cat_full, y, cat_features=cat_cols)
        final_predictions[name] = model.predict_proba(cat_test)[:, 1]
    elif name == 'EXTRA':
        model = make_other_models()['EXTRA']
        model.fit(X_full, y)
        final_predictions[name] = model.predict_proba(X_test)[:, 1]
    elif name == 'RF':
        model = make_other_models()['RF']
        model.fit(X_full, y)
        final_predictions[name] = model.predict_proba(X_test)[:, 1]

test_pred = np.mean(np.column_stack([final_predictions[name] for name in best_models]), axis=1)
test_labels = (test_pred >= 0.5).astype(int)

print('Test predictions:', len(test_labels))
print('Survived:', int(test_labels.sum()))
print('Not survived:', int((1 - test_labels).sum()))
print('Prediction probabilities generated successfully.')

Candidate models for final training: ['CAT_A', 'CAT_B', 'RF']
Best blend found: {'Models': 'CAT_A + CAT_B', 'Accuracy': 0.8406285072951739, 'Weights': 'equal'}
Test predictions: 418
Survived: 139
Not survived: 279
Prediction probabilities generated successfully.


## Experiment 22 conclusion

This experiment compares several strong model families using repeated out-of-fold validation and tests whether their errors can be reduced through blending or stacking.

The final Kaggle submission should only be created after reviewing the validation results.